In [1]:
import sys
import os

# Set working directory to project root
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
os.chdir(project_root)
if project_root not in sys.path:
    sys.path.append(project_root)

import pandas as pd
from src.data.spark_pipeline import get_spark_session
from src.data.databricks_client import DatabricksClient
from src.models.regression import StockPriceRegressor
from src.models.classification import StockClassifier
from src.models.explainability import ModelExplainer

print("Phase 3 imports successful!")

Phase 3 imports successful!


In [2]:
# 1. Initialize sessions & client
spark = get_spark_session()
db_client = DatabricksClient()

# 2. Read stored indicators and sentiment tables
indicators_df = db_client.read_dataset(spark, table_name="aapl_indicators").toPandas()
sentiment_df = db_client.read_dataset(spark, table_name="aapl_sentiment").toPandas()

# 3. Convert timestamps and align datasets on date
indicators_df["date"] = pd.to_datetime(indicators_df["date"])
sentiment_df["date"] = pd.to_datetime(sentiment_df["timestamp"]).dt.date
sentiment_df["date"] = pd.to_datetime(sentiment_df["date"])

# Aggregate daily sentiment score averages
daily_sentiment = (
    sentiment_df.groupby(["date", "ticker"])[
        ["neg_score", "neu_score", "pos_score", "compound_score"]
    ]
    .mean()
    .reset_index()
)

# Merge Technical Features with Sentiment Features
df = pd.merge(indicators_df, daily_sentiment, on=["date", "ticker"], how="left").fillna(0)
df.head()

[2026-09-23 17:51:48] [INFO] [stonks_maker]: Reading dataset locally from /home/jovyan/work/data/processed/aapl_indicators
[2026-09-23 17:51:52] [INFO] [stonks_maker]: Reading dataset locally from /home/jovyan/work/data/processed/aapl_sentiment


,date,ticker,open,high,low,close,adj_close,volume,sma_20,sma_50,bollinger_upper,bollinger_lower,rsi_14,neg_score,neu_score,pos_score,compound_score
0,2025-09-23,AAPL,254.938267,256.392885,252.646729,253.493591,253.493591,60275200,253.493591,253.493591,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0
1,2025-09-24,AAPL,254.280703,254.798793,250.116078,251.381409,251.381409,42303700,252.437500,252.437500,255.424577,249.450423,0.000000,0.0,0.0,0.0,0.0
2,2025-09-25,AAPL,252.278103,256.223536,250.783624,255.924622,255.924622,55202100,253.599874,253.599874,258.146815,249.052933,68.263574,0.0,0.0,0.0,0.0
3,2025-09-26,AAPL,253.164826,256.651945,252.845996,254.519821,254.519821,46076300,253.829861,253.829861,257.654703,250.005019,56.366007,0.0,0.0,0.0,0.0
4,2025-09-29,AAPL,253.623118,254.061501,252.078819,253.493591,253.493591,40127700,253.762607,253.762607,257.088644,250.436569,49.999972,0.0,0.0,0.0,0.0


In [3]:
# --- 1. Train Target Price Regressor ---
regressor = StockPriceRegressor()
X_reg, y_reg = regressor.prepare_data(df, target_horizon=1)

# Split Train/Test (Time Series split)
split_idx = int(len(X_reg) * 0.8)
X_train_r, X_test_r = X_reg.iloc[:split_idx], X_reg.iloc[split_idx:]
y_train_r, y_test_r = y_reg.iloc[:split_idx], y_reg.iloc[split_idx:]

regressor.train(X_train_r, y_train_r)
reg_metrics = regressor.evaluate(X_test_r, y_test_r)
regressor.save_model("models/regressor.joblib")

# --- 2. Train Classifiers ---
classifier = StockClassifier()
prepared_clf_df = classifier.prepare_data(df)

feature_cols = regressor.feature_names
X_clf = prepared_clf_df[feature_cols]
y_dir = prepared_clf_df["target_direction"]
y_strat = prepared_clf_df["target_strategy"]

X_train_c, X_test_c = X_clf.iloc[:split_idx], X_clf.iloc[split_idx:]
classifier.train_direction(X_train_c, y_dir.iloc[:split_idx])
classifier.train_strategy(X_train_c, y_strat.iloc[:split_idx])

clf_metrics = classifier.evaluate(X_test_c, y_dir.iloc[split_idx:], y_strat.iloc[split_idx:])
classifier.save_models()

print("Model Training Complete!")

[2026-09-23 17:51:53] [INFO] [stonks_maker]: Training Stock Price Regressor...
[2026-09-23 17:51:53] [INFO] [stonks_maker]: Regressor training completed.
[2026-09-23 17:51:53] [INFO] [stonks_maker]: Regression Evaluation Metrics: {'rmse': 14.51179846331689, 'mae': 11.330491224528064, 'r2': -0.6297886768974625}
[2026-09-23 17:51:53] [INFO] [stonks_maker]: Regressor saved to models/regressor.joblib
[2026-09-23 17:51:53] [INFO] [stonks_maker]: Training Movement Direction Classifier (UP/DOWN)...
[2026-09-23 17:51:53] [INFO] [stonks_maker]: Training Strategy Classifier (Day vs Swing)...
[2026-09-23 17:51:53] [INFO] [stonks_maker]: Classification Metrics: {'direction_accuracy': 0.44, 'direction_f1': 0.43264367816091953, 'strategy_accuracy': 0.44, 'strategy_f1': 0.4373076923076923}
[2026-09-23 17:51:53] [INFO] [stonks_maker]: Classifier models saved successfully.
Model Training Complete!


In [4]:
# Compute SHAP explanation on the latest sample instance
sample_instance = X_test_r.tail(1)
explainer = ModelExplainer(regressor.model, regressor.feature_names)
explanation = explainer.explain_instance(sample_instance)

print("SHAP Feature Contributions for Latest Prediction:")
print(explanation)

SHAP Feature Contributions for Latest Prediction:
{'base_value': 272.17988372802733, 'feature_contributions': {'open': 0.9290802959813227, 'high': 3.3985244763024762, 'low': 1.2108948333204157, 'close': 15.403171890793809, 'adj_close': 15.463666130694813, 'volume': 0.35442392921781346, 'sma_20': -0.32446490132152883, 'sma_50': 2.34492927706249, 'bollinger_upper': 1.9806945407192356, 'bollinger_lower': -1.3126738672481648, 'rsi_14': 0.4904768855604532, 'neg_score': 0.0, 'neu_score': 0.0, 'pos_score': 0.0, 'compound_score': 0.0}}
